# Imports

In [15]:
!uv pip install -r ./requirements.txt

Using Python 3.13.13 environment at: C:\Users\dante\miniconda3\envs\llm-engineering
Checked 7 packages in 21ms


In [16]:
from __future__ import annotations

import time
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Optional
from urllib.parse import urljoin
import os
from dotenv import load_dotenv

import yaml
from bs4 import BeautifulSoup
from IPython.display import display, Markdown
from openai import OpenAI

from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.remote.webdriver import WebDriver
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.chrome.service import Service as ChromeService

from selenium.webdriver.edge.options import Options as EdgeOptions
from selenium.webdriver.edge.service import Service as EdgeService

from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.firefox.service import Service as FirefoxService

# Keys

In [17]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


# Configs

In [18]:
client = OpenAI()


DEFAULT_USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/117.0.0.0 Safari/537.36"
)

In [19]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [20]:
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

# Classes

In [21]:
@dataclass
class WebDriverConfig:
    browser: str = "chrome"
    headless: bool = True
    driver_path: Optional[str] = None
    browser_binary_path: Optional[str] = None

    page_load_timeout_seconds: int = 30
    wait_timeout_seconds: int = 10
    sleep_after_load_seconds: float = 1.0

    window_size: str = "1920,1080"
    user_agent: str = DEFAULT_USER_AGENT
    disable_images: bool = True
    extra_args: list[str] = field(default_factory=list)

In [22]:
class Website:
    """
    Render and parse a website once.

    This avoids fetching/parsing the same page twice when you need both
    text contents and links.
    """

    def __init__(
        self,
        url: str,
        config_path: str | Path = "webdriver_config.yaml",
        max_chars: int = 2_000,
        wait_for_css: str = "body",
    ):
        self.url = url
        self.config_path = config_path
        self.max_chars = max_chars
        self.wait_for_css = wait_for_css

        self.html = fetch_rendered_html(
            url=url,
            config_path=config_path,
            wait_for_css=wait_for_css,
        )

        self.soup = BeautifulSoup(self.html, "html.parser")
        self.title = self._extract_title()
        self.text = self._extract_text()
        self.links = self._extract_links()

    def _extract_title(self) -> str:
        if self.soup.title:
            return self.soup.title.get_text(" ", strip=True)
        return "No title found"

    def _extract_text(self) -> str:
        body = self.soup.body or self.soup

        for irrelevant in body(["script", "style", "img", "input", "noscript", "svg"]):
            irrelevant.decompose()

        return body.get_text(separator="\n", strip=True)

    def _extract_links(self) -> list[str]:
        links: list[str] = []

        for link in self.soup.find_all("a"):
            href = link.get("href")

            if not href:
                continue

            href = href.strip()

            if href.startswith(("javascript:", "mailto:", "tel:", "#")):
                continue

            links.append(urljoin(self.url, href))

        return links

    def contents(self) -> str:
        return f"{self.title}\n\n{self.text}"[: self.max_chars]

# Functions

In [23]:
def load_webdriver_config(config_path: str | Path = "webdriver_config.yaml") -> WebDriverConfig:
    """
    Load Selenium webdriver config from YAML.
    If the file does not exist, sensible defaults are used.
    """
    config_path = Path(config_path)
    defaults = asdict(WebDriverConfig())

    if not config_path.exists():
        return WebDriverConfig()

    with config_path.open("r", encoding="utf-8") as file:
        loaded = yaml.safe_load(file) or {}

    merged = {**defaults, **loaded}
    return WebDriverConfig(**merged)

In [24]:
def create_driver(config: WebDriverConfig) -> WebDriver:
    """
    Create a Selenium webdriver based on the YAML config.
    Defaults to Chrome in headless mode.
    """
    browser = config.browser.lower().strip()

    if browser == "chrome":
        options = ChromeOptions()

        if config.headless:
            options.add_argument("--headless=new")

        options.add_argument(f"--window-size={config.window_size}")
        options.add_argument(f"--user-agent={config.user_agent}")

        if config.browser_binary_path:
            options.binary_location = config.browser_binary_path

        if config.disable_images:
            options.add_experimental_option(
                "prefs",
                {"profile.managed_default_content_settings.images": 2},
            )

        for arg in config.extra_args:
            options.add_argument(arg)

        service = (
            ChromeService(executable_path=config.driver_path)
            if config.driver_path
            else ChromeService()
        )

        return webdriver.Chrome(service=service, options=options)

    if browser == "edge":
        options = EdgeOptions()

        if config.headless:
            options.add_argument("--headless=new")

        options.add_argument(f"--window-size={config.window_size}")
        options.add_argument(f"--user-agent={config.user_agent}")

        if config.browser_binary_path:
            options.binary_location = config.browser_binary_path

        for arg in config.extra_args:
            options.add_argument(arg)

        service = (
            EdgeService(executable_path=config.driver_path)
            if config.driver_path
            else EdgeService()
        )

        return webdriver.Edge(service=service, options=options)

    if browser == "firefox":
        options = FirefoxOptions()

        if config.headless:
            options.add_argument("-headless")

        if config.browser_binary_path:
            options.binary_location = config.browser_binary_path

        if config.user_agent:
            options.set_preference("general.useragent.override", config.user_agent)

        if config.disable_images:
            options.set_preference("permissions.default.image", 2)

        for arg in config.extra_args:
            options.add_argument(arg)

        service = (
            FirefoxService(executable_path=config.driver_path)
            if config.driver_path
            else FirefoxService()
        )

        return webdriver.Firefox(service=service, options=options)

    raise ValueError(
        f"Unsupported browser: {config.browser!r}. "
        "Use 'chrome', 'edge', or 'firefox'."
    )

In [25]:
def fetch_rendered_html(
    url: str,
    config_path: str | Path = "webdriver_config.yaml",
    wait_for_css: str = "body",
) -> str:
    """
    Render a JavaScript-heavy page with Selenium and return the final HTML.
    """
    config = load_webdriver_config(config_path)
    driver = create_driver(config)

    try:
        driver.set_page_load_timeout(config.page_load_timeout_seconds)
        driver.get(url)

        try:
            WebDriverWait(driver, config.wait_timeout_seconds).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, wait_for_css))
            )
        except TimeoutException:
            # Return whatever was rendered so far.
            pass

        if config.sleep_after_load_seconds > 0:
            time.sleep(config.sleep_after_load_seconds)

        return driver.page_source

    finally:
        driver.quit()

In [26]:
def fetch_website_contents(
    url: str,
    config_path: str | Path = "webdriver_config.yaml",
    max_chars: int = 2_000,
) -> str:
    """
    Return the rendered title and text contents of the website.
    """
    website = Website(url=url, config_path=config_path, max_chars=max_chars)
    return website.contents()


def fetch_website_links(
    url: str,
    config_path: str | Path = "webdriver_config.yaml",
) -> list[str]:
    """
    Return the rendered links from the website.
    """
    website = Website(url=url, config_path=config_path)
    return website.links


def summarize(
    url: str,
    config_path: str | Path = "webdriver_config.yaml",
) -> str:
    website = fetch_website_contents(url=url, config_path=config_path)

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages_for(website),
    )

    return response.choices[0].message.content


def display_summary(
    url: str,
    config_path: str | Path = "webdriver_config.yaml",
) -> None:
    summary = summarize(url=url, config_path=config_path)
    display(Markdown(summary))

In [27]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

# Execution

In [28]:
display_summary("https://openai.com")

# OpenAI Website: The AI Circus You Can't Stop Watching

Welcome to OpenAI’s digital playground where the future is now and it's mostly about GPT-5.5—because why settle for less when you can have 5.5? They just dropped a fresh batch of features like ChatGPT Images 2.0 and workspace agents, probably to make your AI sidekick as multitasking as your toddler.

**Recent Headlines That’ll Blow Your Mind or At Least Make You Raise an Eyebrow:**
- Advanced Account Security launched on April 30, 2026. Because robots hacking robots is apparently a thing now.
- Mysterious goblins explained on April 29. Spoiler: probably an allegory for something techy.
- OpenAI’s models, Codex, and Managed Agents are now hanging out on AWS (April 28). The AI party’s getting bigger.
- Microsoft and OpenAI are still BFFs, moving into the "next phase" (April 27), probably plotting world domination with Excel macros.
- Ethical mumbo jumbo about "Our Principles" surfaced April 26. Serious faces engaged!

You can also have ChatGPT do everything from planning surf trips to decoding obscure languages or even drawing a mini Aussie diver—because why not?

In sum: AI gets cooler, securer, and more useful, while goblins and corporate alliances keep the plot thick. Grab your popcorn.